# Mercadona scraper — estructura común final

Flujo:

1. **Scraping** de las 20 búsquedas desde la web de Mercadona, con reintentos automáticos.
2. **RAW:** todas las apariciones se unen y se guardan en un único CSV.
3. **Procesado:** deduplicación, normalización de cantidades/precios por unidad y eliminación de filas sin precio o unidad comparable.
4. **PROCESSED:** un único CSV final limpio.

Archivos:
- `data/raw/mercadona_products_raw.csv`
- `data/processed/mercadona_products.csv`

In [40]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

HOME_URL = "https://tienda.mercadona.es/"
POSTAL_CODE = "28911"
WAIT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

In [41]:
current = Path.cwd().resolve()
BASE_DIR = current.parent if current.name == "scrapers" else current

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "mercadona_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "mercadona_products.csv"

print("RAW:", RAW_PATH)
print("PROCESSED:", PROCESSED_PATH)

RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\mercadona_products_raw.csv
PROCESSED: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\mercadona_products.csv


In [42]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies_if_needed(driver):
    try:
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(., 'Aceptar')]")
            )
        )
        button.click()
        print("Cookies aceptadas.")
        time.sleep(0.5)

    except TimeoutException:
        pass


def enter_postal_code(driver):
    postal_input = WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "input[type='text']")
        )
    )

    postal_input.clear()
    postal_input.send_keys(POSTAL_CODE)
    postal_input.send_keys(Keys.ENTER)

    print(f"Código postal introducido: {POSTAL_CODE}")

    WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'input[placeholder*="Buscar"]')
        )
    )

    time.sleep(1)


def ensure_store_ready(driver):
    search_inputs = driver.find_elements(
        By.CSS_SELECTOR,
        'input[placeholder*="Buscar"]'
    )

    if any(
        el.is_displayed() and el.is_enabled()
        for el in search_inputs
    ):
        return

    enter_postal_code(driver)


def go_to_home(driver):
    driver.get(HOME_URL)

    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script(
            "return document.readyState"
        ) == "complete"
    )

    accept_cookies_if_needed(driver)
    ensure_store_ready(driver)


def assert_no_security_challenge(driver):
    text = driver.find_element(By.TAG_NAME, "body").text.lower()

    messages = [
        "confirma que no eres un robot",
        "captcha",
        "access denied",
    ]

    if any(message in text for message in messages):
        raise RuntimeError("Posible bloqueo detectado.")


In [43]:
def extract_products(driver, searched_item):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select('div[data-testid="product-cell"]')

    rows = []
    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        name_el = card.select_one(
            '[data-testid="product-cell-name"]'
        )
        size_el = card.select_one(
            ".product-format .sr-only"
        )
        price_el = card.select_one(
            '[data-testid="product-price"]'
        )
        price_type_el = card.select_one(
            ".product-price__extra-price"
        )
        image_el = card.select_one("img")

        if not name_el or not price_el:
            continue

        price_raw = price_el.get_text(strip=True)

        rows.append({
            "searched_item": searched_item,
            "supermarket": "Mercadona",
            "name": name_el.get_text(strip=True),
            "quantity_raw": (
                size_el.get_text(strip=True)
                if size_el
                else None
            ),
            "price_raw": price_raw,
            "price_type": (
                price_type_el.get_text(strip=True)
                if price_type_el
                else None
            ),
            "price": (
                price_raw
                .replace("€", "")
                .replace(",", ".")
                .strip()
            ),
            "image_url": (
                image_el.get("src")
                if image_el
                else None
            ),
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

    return df


def find_search_box(driver):
    return WebDriverWait(driver, WAIT).until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, 'input[placeholder*="Buscar"]')
        )
    )


def search_product(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        lambda d: len(
            d.find_elements(
                By.CSS_SELECTOR,
                'div[data-testid="product-cell"]'
            )
        ) > 0
    )

    assert_no_security_challenge(driver)
    time.sleep(1.5)

    return extract_products(driver, term)

In [44]:
driver = start_driver()
accept_cookies_if_needed(driver)
enter_postal_code(driver)

all_results = []
summary = []

for i, term in enumerate(SEARCH_TERMS, start=1):
    success = False
    started = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 2):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(SEARCH_TERMS)}] {term} · intento {attempt}")

        try:
            go_to_home(driver)
            df_term = search_product(driver, term)

            extracted = len(df_term)

            if extracted == 0:
                raise RuntimeError("La búsqueda no devolvió productos.")

            all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "rows_extracted": extracted,
                "attempts": attempt,
                "status": "OK",
                "seconds": round(time.monotonic() - started, 2),
            })

            print(f"Estado: OK · {extracted} productos")
            success = True
            break

        except Exception as exc:
            print(f"Intento {attempt} fallido: {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES + 1:
                time.sleep(2)

    if not success:
        summary.append({
            "searched_item": term,
            "rows_extracted": 0,
            "attempts": MAX_RETRIES + 1,
            "status": "ERROR",
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)

Cookies aceptadas.
Código postal introducido: 28911

[1/20] leche · intento 1
Estado: OK · 59 productos

[2/20] huevos · intento 1
Estado: OK · 60 productos

[3/20] manzana · intento 1
Estado: OK · 31 productos

[4/20] plátano · intento 1
Estado: OK · 20 productos

[5/20] tomate · intento 1
Estado: OK · 60 productos

[6/20] arroz · intento 1
Estado: OK · 59 productos

[7/20] pasta · intento 1
Estado: OK · 60 productos

[8/20] pan · intento 1
Estado: OK · 60 productos

[9/20] pollo · intento 1
Estado: OK · 60 productos

[10/20] carne picada · intento 1
Estado: OK · 53 productos

[11/20] salmón · intento 1
Estado: OK · 31 productos

[12/20] yogur · intento 1
Estado: OK · 60 productos

[13/20] queso · intento 1
Estado: OK · 60 productos

[14/20] agua · intento 1
Estado: OK · 60 productos

[15/20] zumo · intento 1
Estado: OK · 60 productos

[16/20] atún · intento 1
Estado: OK · 47 productos

[17/20] garbanzos · intento 1
Estado: OK · 13 productos

[18/20] aceite · intento 1
Estado: OK · 60

,searched_item,rows_extracted,attempts,status,seconds
0,leche,59,1,OK,12.59
1,huevos,60,1,OK,9.20
2,manzana,31,1,OK,9.50
3,plátano,20,1,OK,8.89
4,tomate,60,1,OK,10.05
5,arroz,59,1,OK,9.56
6,pasta,60,1,OK,11.33
7,pan,60,1,OK,9.70
8,pollo,60,1,OK,8.91
9,carne picada,53,1,OK,8.95


## RAW — unir todas las búsquedas y guardar un único archivo

In [45]:
if not all_results:
    raise RuntimeError("No se extrajo ninguna búsqueda.")

df_mercadona_raw = pd.concat(
    all_results,
    ignore_index=True
)

df_mercadona_raw.to_csv(
    RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Filas RAW:", len(df_mercadona_raw))
print("RAW guardado:", RAW_PATH)

Filas RAW: 993
RAW guardado: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\mercadona_products_raw.csv


## PROCESSED — deduplicar, normalizar y eliminar filas incompletas

In [46]:
def parse_size(size_text):
    if pd.isna(size_text):
        return None, None

    text = str(size_text).lower().strip().replace(",", ".")

    replacements = {
        "mililitros": "ml",
        "mililitro": "ml",
        "litros": "l",
        "litro": "l",
        "kilogramos": "kg",
        "kilogramo": "kg",
        "kilos": "kg",
        "kilo": "kg",
        "gramos": "g",
        "gramo": "g",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    match = re.search(
        r"(\d+).*?x\s*([\d.]+)\s*(ml|l|kg|g)",
        text
    )

    if match:
        count = int(match.group(1))
        size = float(match.group(2))
        raw_unit = match.group(3)

        if raw_unit == "ml":
            return count * size / 1000, "LITRO"
        if raw_unit == "l":
            return count * size, "LITRO"
        if raw_unit == "g":
            return count * size / 1000, "KILO"
        if raw_unit == "kg":
            return count * size, "KILO"

    match = re.search(
        r"(\d+)\s*(?:unidades|unidad|uds|ud)\b",
        text
    )

    if match:
        return float(match.group(1)), "UNIDAD"

    match = re.search(
        r"([\d.]+)\s*(ml|l|kg|g)\b",
        text
    )

    if match:
        size = float(match.group(1))
        raw_unit = match.group(2)

        if raw_unit == "ml":
            return size / 1000, "LITRO"
        if raw_unit == "l":
            return size, "LITRO"
        if raw_unit == "g":
            return size / 1000, "KILO"
        if raw_unit == "kg":
            return size, "KILO"

    match = re.search(
        r"(\d+)\s*(rollos?|servicios?|sobres?)\b",
        text
    )

    if match:
        quantity = float(match.group(1))
        raw_unit = match.group(2)

        if raw_unit.startswith("rollo"):
            return quantity, "ROLLO"
        if raw_unit.startswith("servicio"):
            return quantity, "SERVICIO"
        if raw_unit.startswith("sobre"):
            return quantity, "SOBRE"

    return None, None

In [47]:
df_mercadona_processed = (
    df_mercadona_raw
    .drop_duplicates(
        subset=["name", "quantity_raw", "price"]
    )
    .reset_index(drop=True)
    .copy()
)

parsed = df_mercadona_processed["quantity_raw"].apply(parse_size)

df_mercadona_processed["total_quantity"] = (
    parsed.apply(lambda x: x[0])
)

df_mercadona_processed["unit"] = (
    parsed.apply(lambda x: x[1])
)

df_mercadona_processed["price_per_unit"] = (
    df_mercadona_processed["price"]
    / df_mercadona_processed["total_quantity"]
)

df_mercadona_processed["price_per_unit"] = (
    pd.to_numeric(
        df_mercadona_processed["price_per_unit"],
        errors="coerce"
    )
    .round(2)
)

df_mercadona_processed["postal_code"] = POSTAL_CODE
df_mercadona_processed["warehouse"] = "mad2"

before_drop = len(df_mercadona_processed)

df_mercadona_processed = (
    df_mercadona_processed[
        df_mercadona_processed["price"].notna()
        & df_mercadona_processed["price_per_unit"].notna()
        & df_mercadona_processed["unit"].notna()
    ]
    .reset_index(drop=True)
)

print("Únicos antes de eliminar incompletos:", before_drop)
print("Productos finales:", len(df_mercadona_processed))
print(
    "Eliminados por precio/unidad incompletos:",
    before_drop - len(df_mercadona_processed)
)
print("\nUnidades:")
print(df_mercadona_processed["unit"].value_counts(dropna=False))

Únicos antes de eliminar incompletos: 895
Productos finales: 895
Eliminados por precio/unidad incompletos: 0

Unidades:
unit
KILO        549
LITRO       252
UNIDAD       87
ROLLO         4
SERVICIO      2
SOBRE         1
Name: count, dtype: int64


In [48]:
df_mercadona_processed.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("PROCESSED guardado:", PROCESSED_PATH)

PROCESSED guardado: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\mercadona_products.csv


In [49]:
driver.quit()
print("Driver cerrado.")

Driver cerrado.
